# 🛠️ Notebook 3: Build a Tiny Metrics Collector

In this notebook we build a **mini Prometheus** — a tiny in-memory metrics collector with counters, gauges, and histograms. Then we add **labels** (so we can slice by endpoint), see how labels can blow up your memory (**cardinality**), and finally implement a **bucketed histogram** the way real monitoring systems actually do it.

## Learning objectives
- Implement counters, gauges, and bucketed histograms in <100 lines.
- Add **labels** so a single metric name can carry multiple time series.
- Recognize and avoid the **cardinality explosion** anti-pattern.
- Compute approximate percentiles from histogram buckets (the *Prometheus* way) and contrast with exact percentiles from raw samples (the *teaching* way).
- Plot a latency histogram with matplotlib.
- See why **averaging p99s across instances** is wrong in both directions, and what to do instead.

## Step 1 — A tiny `Registry` with counters, gauges, and raw-sample histograms

For pedagogy we start by storing **every sample** so we can compute exact percentiles. We'll see why real systems don't do this in Step 3.

In [ ]:
import time, random, bisect
from collections import defaultdict

class Registry:
    """Tiny metrics registry. Labels are stored as a tuple of (key, value) pairs."""
    def __init__(self):
        self._counters: dict[tuple, float] = defaultdict(float)
        self._gauges:   dict[tuple, float] = {}
        self._samples:  dict[tuple, list[float]] = defaultdict(list)

    @staticmethod
    def _key(name, labels):
        # Sorted tuple so {a=1,b=2} and {b=2,a=1} share the same series.
        return (name, tuple(sorted((labels or {}).items())))

    # --- counters ---
    def inc(self, name, by=1.0, **labels):
        self._counters[self._key(name, labels)] += by

    # --- gauges ---
    def set(self, name, v, **labels):
        self._gauges[self._key(name, labels)] = v

    # --- histograms (raw samples — easy to reason about, expensive in production) ---
    def observe(self, name, v, **labels):
        self._samples[self._key(name, labels)].append(v)

    def percentile(self, name, p, **labels):
        s = sorted(self._samples.get(self._key(name, labels), []))
        if not s:
            return None
        k = max(0, min(len(s) - 1, int(round(p * (len(s) - 1)))))
        return s[k]

    # --- public accessors (don't poke private attrs from outside) ---
    def samples(self, name, **labels):
        return list(self._samples.get(self._key(name, labels), []))

    def series_count(self):
        return len(self._counters) + len(self._gauges) + len(self._samples)

    def render(self):
        """Prometheus-ish text exposition format."""
        def fmt(name, lbls, extra=()):
            pairs = list(lbls) + list(extra)
            inner = ",".join(f'{k}="{v}"' for k, v in pairs)
            return f"{name}{{{inner}}}" if inner else name

        lines, typed = [], set()

        def maybe_type(name, kind):
            if name not in typed:
                lines.append(f"# TYPE {name} {kind}")
                typed.add(name)

        for (name, lbls), v in self._counters.items():
            maybe_type(name, "counter")
            lines.append(f"{fmt(name, lbls)} {v}")
        for (name, lbls), v in self._gauges.items():
            maybe_type(name, "gauge")
            lines.append(f"{fmt(name, lbls)} {v}")
        for name, lbls in self._samples:
            maybe_type(name, "summary")
            for q in (0.5, 0.95, 0.99):
                v = self.percentile(name, q, **dict(lbls))
                lines.append(f'{fmt(name, lbls, [("quantile", q)])} {v:.3f}')
        return "\n".join(lines)

## Step 2 — Use it: requests with **labels**

A single metric name (`http_requests_total`) becomes many *time series* once you add labels — one per unique label combination. That's incredibly powerful (you can break down by endpoint, status, method, region…) but also dangerous, as we'll see.

In [ ]:
m = Registry()
random.seed(0)

endpoints = ["/checkout", "/cart", "/search"]
for _ in range(5000):
    ep = random.choice(endpoints)
    # /checkout is intentionally slower
    base = 200 if ep == "/checkout" else 60
    latency = random.lognormvariate(0, 0.6) * base
    status = "200" if random.random() > 0.01 else "500"
    m.inc("http_requests_total", endpoint=ep, status=status)
    m.observe("http_latency_ms", latency, endpoint=ep)

m.set("memory_in_use_mb", 412)

print("Total time series stored:", m.series_count())
print()
for ep in endpoints:
    p95 = m.percentile("http_latency_ms", 0.95, endpoint=ep)
    print(f"  {ep:<10} p95 = {p95:6.1f} ms")

# Labels are order-insensitive: {a,b} and {b,a} must be the SAME series, not two.
probe = Registry()
probe.inc("x", endpoint="/a", status="200")
probe.inc("x", status="200", endpoint="/a")
assert probe.series_count() == 1

# /checkout was seeded to be ~3x slower; the label breakdown has to show that.
assert m.percentile("http_latency_ms", 0.95, endpoint="/checkout") > \
       2 * m.percentile("http_latency_ms", 0.95, endpoint="/cart")

print("\n--- what a /metrics endpoint would return (excerpt) ---")
print("\n".join(probe.render().splitlines()))

## Step 3 — ⚠️ The cardinality trap

Each unique combination of `(metric_name, labels)` is a separate **time series** that the monitoring backend must store, index, and query. Beginners often add a label like `user_id` or `request_id` thinking "more detail is better!" — and accidentally create *millions* of series.

> **Rule of thumb:** label values must come from a **small, bounded** set (`endpoint`, `status_code`, `region`, `method`). Never use unbounded values like user IDs, email addresses, request IDs, URLs with parameters, etc.

Let's see the difference.

In [ ]:
bad = Registry()
for i in range(2000):
    # ❌ user_id is unbounded -> one series per user.
    bad.inc("http_requests_total", endpoint="/checkout", user_id=f"user-{i}")

good = Registry()
for i in range(2000):
    # ✅ Bucket the user into a small set of plan tiers.
    plan = random.choice(["free", "pro", "enterprise"])
    good.inc("http_requests_total", endpoint="/checkout", plan=plan)

print(f"❌ bad  registry: {bad.series_count():>5} series  (will OOM in production)")
print(f"✅ good registry: {good.series_count():>5} series  (fine)")

# One series per user vs one per plan tier: the label's *value space* is what costs
# money, not the number of requests.
assert bad.series_count() == 2000
assert good.series_count() <= 3
print(f"\nSame 2000 requests, {bad.series_count() // good.series_count()}x the series.")
print("A Prometheus instance is sized in millions of series; a single unbounded")
print("label on a busy endpoint gets you there in an afternoon.")

For *per-user* breakdowns, use **logs or traces** — they're built for high-cardinality data. Metrics are for low-cardinality aggregates.

## Step 4 — Bucketed histograms (the *real* Prometheus way)

Storing every raw sample (Step 1) is great for teaching but doesn't scale: at 100k req/s your samples list explodes. Real systems use **bucketed histograms**: pre-defined upper bounds (`le` = "less than or equal to"), and you just count how many observations fell into each bucket.

Important Prometheus details:
- Buckets are **cumulative** — `le=0.5` includes everything `≤ 0.5` *and* everything in smaller buckets.
- Each bucket is a counter; you also keep `_sum` and `_count`.
- Percentiles from buckets are **approximations** (Prometheus's `histogram_quantile` does linear interpolation inside the right bucket).

Don't expect bucket-based p95 to match raw-sample p95 exactly — that's the trade-off you make to save memory.

In [ ]:
class BucketedHistogram:
    """A Prometheus-style histogram: cumulative bucket counts + _sum + _count."""
    def __init__(self, buckets):
        # Always include +inf as the last bucket so every observation lands somewhere.
        self.bounds = sorted(buckets) + [float("inf")]
        self.counts = [0] * len(self.bounds)   # CUMULATIVE counts
        self.sum = 0.0
        self.count = 0

    def observe(self, v):
        self.sum += v
        self.count += 1
        # Increment every bucket whose upper bound is >= v (cumulative).
        idx = bisect.bisect_left(self.bounds, v)
        for i in range(idx, len(self.bounds)):
            self.counts[i] += 1

    def quantile(self, q):
        """Approximate quantile via linear interpolation inside the matching bucket."""
        if self.count == 0:
            return None
        target = q * self.count
        prev_count, prev_bound = 0, 0.0
        for c, b in zip(self.counts, self.bounds):
            if c >= target:
                if b == float("inf"):
                    return prev_bound
                bucket_size = c - prev_count
                if bucket_size == 0:
                    return b
                # Linear interpolation between prev_bound and b.
                frac = (target - prev_count) / bucket_size
                return prev_bound + frac * (b - prev_bound)
            prev_count, prev_bound = c, b
        return self.bounds[-2]

    def merge(self, other):
        """Add another histogram's counts into this one.

        This is the property that makes bucketed histograms special: two servers'
        histograms can be summed into one that is exactly the histogram you would
        have got by observing all the samples in one place. You cannot do this
        with pre-computed percentiles.
        """
        assert self.bounds == other.bounds, "histograms must share bucket bounds"
        merged = BucketedHistogram(self.bounds[:-1])
        merged.counts = [a + b for a, b in zip(self.counts, other.counts)]
        merged.sum = self.sum + other.sum
        merged.count = self.count + other.count
        return merged

    def render(self, name):
        out = [f"# TYPE {name} histogram"]
        for c, b in zip(self.counts, self.bounds):
            le = "+Inf" if b == float("inf") else f"{b:g}"
            out.append(f'{name}_bucket{{le="{le}"}} {c}')
        out.append(f"{name}_sum {self.sum:.3f}")
        out.append(f"{name}_count {self.count}")
        return "\n".join(out)


# Buckets in milliseconds — pick them to cover your expected range.
hist = BucketedHistogram([5, 10, 25, 50, 100, 250, 500, 1000, 2500])

raw_samples = []
random.seed(0)
for _ in range(5000):
    latency = random.lognormvariate(4, 0.6)
    hist.observe(latency)
    raw_samples.append(latency)

print(hist.render("http_latency_ms"))
print()
print("Approx (bucketed) vs Exact (raw) percentiles:")
for q in (0.5, 0.95, 0.99):
    raw_sorted = sorted(raw_samples)
    exact = raw_sorted[int(q * (len(raw_sorted) - 1))]
    approx = hist.quantile(q)
    print(f"  p{int(q*100)}: bucketed ≈ {approx:7.1f} ms   raw = {exact:7.1f} ms")

# Bucket counts are cumulative and the last one must hold every observation.
assert hist.counts == sorted(hist.counts)
assert hist.counts[-1] == hist.count == len(raw_samples)
# The approximation has to stay in the right ballpark — within one bucket width.
for q in (0.5, 0.95, 0.99):
    exact = sorted(raw_samples)[int(q * (len(raw_samples) - 1))]
    assert 0.7 * exact < hist.quantile(q) < 1.4 * exact, q

Notice the bucketed values are **close but not equal** to the exact percentiles. That's the trade-off:

|  | Memory | Accuracy | Aggregation across servers |
|---|---|---|---|
| Raw samples | 💸 grows forever | exact | hard |
| Bucketed | 💚 fixed (one int per bucket) | approximate | easy (just sum buckets) |

Production monitoring systems pick bucketed because (a) memory is bounded and (b) you can sum the bucket counters from 100 servers and *still* compute a global percentile.

## Step 5 — 🚫 The one arithmetic mistake everybody makes: averaging percentiles

You have ten instances of a service. Each one exports its own `p99`. Your dashboard
shows one number, so it does the obvious thing:

```promql
avg(http_latency_p99)     # ❌ this is not the p99 of anything
```

**A percentile is not a mean, and means of percentiles are not percentiles.** There
is no weighting of per-instance p99s that recovers the fleet p99, because the p99
throws away exactly the information you would need — how many samples sat where.

The only correct way is to **merge the distributions first, then take the
percentile**. That is precisely why Prometheus histograms expose cumulative bucket
counters: `sum(rate(..._bucket[5m])) by (le)` adds the buckets across instances, and
`histogram_quantile()` reads the percentile off the merged shape.

Let's build two fleets that both defeat the averaging shortcut — in opposite
directions.

In [ ]:
BUCKETS = [5, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000]

def instance_histogram(samples):
    h = BucketedHistogram(BUCKETS)
    for s in samples:
        h.observe(s)
    return h

def fleet_p99(histograms):
    """The RIGHT way: merge the bucket counts, then read the quantile off the total."""
    merged = histograms[0]
    for h in histograms[1:]:
        merged = merged.merge(h)
    return merged.quantile(0.99), merged

def average_of_p99s(histograms):
    """The WRONG way: let every instance compute its own p99, then take the mean."""
    return sum(h.quantile(0.99) for h in histograms) / len(histograms)


# --- Fleet A: one instance is sick, and it carries a normal share of traffic ---
healthy = [instance_histogram([40.0] * 990 + [90.0] * 10) for _ in range(9)]
sick    = instance_histogram([3000.0] * 1000)
fleet_a = healthy + [sick]

true_p99, merged_a = fleet_p99(fleet_a)
avg_p99 = average_of_p99s(fleet_a)
print("Fleet A — 1 of 10 instances is timing out, all serving 1000 req each")
print(f"  avg of per-instance p99s : {avg_p99:8.0f} ms   ❌")
print(f"  true fleet-wide p99      : {true_p99:8.0f} ms   ✅")
print(f"  -> the dashboard understates the tail by {true_p99/avg_p99:.0f}x")
assert true_p99 > 5 * avg_p99

In [ ]:
# --- Fleet B: the sick instance is nearly idle (a canary, or a draining pod) ---
busy   = [instance_histogram([40.0] * 990 + [90.0] * 10) for _ in range(9)]
canary = instance_histogram([3000.0] * 10)        # only 10 requests, all slow
fleet_b = busy + [canary]

true_p99_b, merged_b = fleet_p99(fleet_b)
avg_p99_b = average_of_p99s(fleet_b)
print("Fleet B — same sick instance, but it only served 10 of 8,920 requests")
print(f"  avg of per-instance p99s : {avg_p99_b:8.0f} ms   ❌")
print(f"  true fleet-wide p99      : {true_p99_b:8.0f} ms   ✅")
print(f"  -> the same dashboard now OVERstates the tail by {avg_p99_b/true_p99_b:.0f}x")
assert avg_p99_b > 3 * true_p99_b

print("\nSame arithmetic, opposite error. Averaging p99s is not 'approximately right' —")
print("its sign depends on traffic you cannot see from the p99 alone.")

# Merging is lossless in the way that matters: total count and total sum are exact,
# so the merged histogram is the one you would have recorded centrally.
assert merged_b.count == sum(h.count for h in fleet_b)
assert abs(merged_b.sum - sum(h.sum for h in fleet_b)) < 1e-6

### What to do instead

| You have | Do this |
|---|---|
| Prometheus histograms | `histogram_quantile(0.99, sum(rate(x_bucket[5m])) by (le))` — sum **buckets**, then quantile |
| Prometheus *summaries* | you can't aggregate them at all; the per-instance quantiles are pre-computed. Switch to a histogram. |
| A hosted APM | check whether its "p99 across hosts" merges distributions or averages. Many average. |
| Truly need a single number to average | average the **rate of requests over your SLO threshold** (a counter), not the percentile |

That last row is the practical escape hatch: `count(latency > 300ms) / count(*)` is a
ratio of counters, and ratios of counters **do** aggregate correctly. It is also
exactly the SLI shape from Notebook 2 — which is not a coincidence. SLIs are defined
as "good events / total events" partly *because* that definition survives being summed
across a fleet.

## Step 6 — Visualize the latency distribution

In [ ]:
import matplotlib.pyplot as plt

samples = m.samples("http_latency_ms", endpoint="/checkout")
plt.figure(figsize=(8, 4))
plt.hist(samples, bins=60, color="steelblue", edgecolor="white")
for p, color in [(0.5, "green"), (0.95, "orange"), (0.99, "red")]:
    v = m.percentile("http_latency_ms", p, endpoint="/checkout")
    plt.axvline(v, color=color, linestyle="--", label=f"p{int(p*100)} = {v:.0f} ms")
plt.title("HTTP request latency — /checkout")
plt.xlabel("latency (ms)")
plt.ylabel("count")
plt.legend()
plt.tight_layout()
plt.show()

## ✅ Recap

A real metrics system (Prometheus, Datadog, OpenTelemetry…) is doing roughly the same thing as this notebook — just with:

- **bucketed histograms** instead of full sample arrays (saves memory, aggregates across hosts),
- a **sliding time window** so you see *recent* values, not lifetime totals,
- **pull or push over the network** with a text/binary exposition format,
- **labels** for slicing — kept low-cardinality on purpose.

Two failure modes to carry with you: **never average percentiles** (merge the buckets, then take the quantile), and watch out for the **cardinality trap**: high-cardinality fields (user IDs, request IDs, full URLs) belong in **logs** or **traces**, not in metric labels.

In the final notebook we'll **debug a fake outage** using all three pillars together.